# Reproducing Churchland et al. (2010) from dandiset 000128

**Paper.** Churchland MM, Cunningham JP, Kaufman MT, Ryu SI, Shenoy KV (2010). Cortical preparatory
activity: representation of movement or first cog in a dynamical machine? *Neuron* 68:387-400,
doi:10.1016/j.neuron.2010.09.015.

**Dataset.** DANDI:000128 (MC_Maze, Neural Latents Benchmark), version `0.220113.0400`,
doi:10.48324/dandi.000128/0.220113.0400. The paper cites no version of the dataset (it predates it).
This is the only published version, so it is pinned. The draft (modified 2023-06-20) holds the same two
assets with the same blobs (checked below), so the choice changes nothing the analysis uses.

**Claims file.** `claims/000128/10.1016-j.neuron.2010.09.015.yaml`, written from the accepted author
manuscript in PubMed Central (PMC2991102) with its supplement, and checked by a second reader. The
claims file fixes every method; the labels M1 to M9, T2, S1 and so on are its readings.

**Answers taken up.** A reader answered three questions of this attempt on 2026-10-22
(`questions/000128.yaml`), and the claims file was updated. The proto-trial selection (M1) is now reading
T2 alone: T1, which kept 73.8% of trials and emptied two conditions, is set aside. The tuning measure (M6)
is now reading S1 (the range over conditions) alone: S2, the SD over conditions, is set aside. For
suppfig1-population-averages the criterion now names both readings of the normalising maximum, (a) and
(b), which were already run. The analyses below run T2 and S1 only; T1's count is printed once, for the
record.

The dataset holds one session of monkey Jenkins on the maze task, recorded with two 96-electrode
arrays: the paper's J-array dataset. It holds none of the speed-task data (monkeys A and B), none of the
single-electrode maze recordings (monkey J) and no EMG.

| Claim | Status here |
|---|---|
| recordings-jarray | tested |
| recordings-single-electrode | blocked: data_missing |
| fig2b-example-concordant-tuning | blocked: data_missing (speed task) |
| fig2ce-example-opposing-tuning | blocked: data_missing (speed task) |
| fig3-maze-most-not-maintained | tested |
| fig4d-mean-correlation-jarray | tested |
| fig4abc-mean-correlation-single-electrode | blocked: data_missing |
| fig4-no-separate-populations | tested (J-array only) |
| fig4e-correlation-time-course | tested |
| suppfig1-population-averages | tested |
| fig6-peri-movement-space-best | tested (J-array) |
| fig6-each-dimensionality | tested (J-array) |
| fig6-performance-vs-dimensionality | tested (J-array) |
| prep-dimensionality | tested (J-array) |
| fig6e-summary-across-datasets | blocked: data_missing (A, B, J and EMG) |
| fig7a-shuffled-space-poor | tested (J-array) |
| fig7b-window-start-slide | tested (J-array ratio, reading J2) |
| ap-location-no-effect-on-correlation | blocked: metadata_insufficient (no AP location; array labels wrong); a variant is shown |
| fig7c-anterior-posterior-performance | blocked: metadata_insufficient (as above) |
| fig7d-pd-consistency | tested (J-array) |
| fig8-model-weak-correlation | tested (simulation) |
| fig8-model-pd-perfect | tested (simulation) |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import diptest
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold, cross_val_score
import mcmaze as mm

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
T_START = time.time()
print("dandiset", mm.DANDISET_ID, "version", mm.VERSION)

## Data check

In [ ]:
meta = mm.version_metadata()
pub = {a["path"]: (a["blob"], a["size"]) for a in mm.list_assets(mm.VERSION)}
draft = {a["path"]: (a["blob"], a["size"]) for a in mm.list_assets("draft")}
print("title:", meta["name"])
print("related publications in metadata:", [r.get("identifier") for r in meta.get("relatedResource", [])])
for p, (b, s) in pub.items():
    print(f"  {p}: {s/1e6:.1f} MB, blob {b}")
print("draft holds the same assets with the same blobs:", pub == draft)

nwb, io = mm.open_nwb(mm.TRAIN_PATH)
sess = mm.load_session(nwb)
nwb_test, io_test = mm.open_nwb(mm.TEST_PATH)
tr = sess["trials"]
units = sess["units"]
print("subject:", nwb.subject.subject_id, nwb.subject.species, "| session:", nwb.session_start_time.date(),
      "| related publication in file:", list(nwb.related_publications))
print("train file trials columns:", list(tr.columns))
print("units columns:", nwb.units.colnames)
print("electrodes columns:", list(nwb.electrodes.colnames))
print("behaviour:", list(nwb.processing["behavior"].data_interfaces))

In [ ]:
# what was loaded: counts and ranges of every variable used
print("=== trials (train file) ===")
print("trials:", len(tr), "| split:", tr.split.value_counts().to_dict(), "| success all True:", bool(tr.success.all()))
print("trial_type values:", tr.trial_type.nunique(), "| maze_id values:", tr.maze_id.nunique(),
      "| trial_type <-> maze_id one to one:", tr.groupby("trial_type").maze_id.nunique().max() == 1
      and tr.groupby("maze_id").trial_type.nunique().max() == 1)
print("conditions (trial_type x trial_version):", tr.condition.nunique())
print("trials per condition: min", tr.condition.value_counts().min(), "median", tr.condition.value_counts().median(),
      "max", tr.condition.value_counts().max())
print(pd.crosstab(tr.trial_version, [tr.num_targets, tr.num_barriers > 0]).rename_axis(index="version",
      columns=["targets", "barriers"]))
print("delay (ms): min", tr.delay.min(), "max", tr.delay.max(), "| delay > 400 ms:", (tr.delay > 400).sum(),
      "| negative delay:", (tr.delay < 0).sum())
print("delay equals go - target onset (max |diff| ms):", np.abs((tr.go_cue_time - tr.target_on_time) * 1000 - tr.delay).max().round(6))
print("rt (ms): min", tr.rt.min(), "max", tr.rt.max(), "| rt equals move onset - go (max |diff| ms):",
      np.abs((tr.move_onset_time - tr.go_cue_time) * 1000 - tr.rt).max().round(6))
print("trial start to go cue (s): min %.3f; move onset to trial end (s): min %.3f" % (
      (tr.go_cue_time - tr.start_time).min(), (tr.stop_time - tr.move_onset_time).min()))
print("target positions (mm): x %d to %d, y %d to %d" % (tr.target_x.min(), tr.target_x.max(), tr.target_y.min(), tr.target_y.max()))

print("\n=== units ===")
print("units:", len(units), "| held out:", int(units.heldout.sum()), "| held in:", int((~units.heldout).sum()))
print("spikes per unit: min", units.n_spikes.min(), "median", int(units.n_spikes.median()), "max", units.n_spikes.max())
print("electrode group of each unit, as the units table gives it:", units.electrode_group.value_counts().to_dict())
el = nwb.electrodes.to_dataframe()
print("electrodes table:", el.group_name.value_counts().to_dict(), "| x, y, z all NaN:",
      bool(el[["x", "y", "z"]].isna().all().all()))
print("units' electrode row equals the unit id's channel digits minus 1 for every unit:",
      bool((units.electrode_row == units.id_channel - 1).all()))
print("leading digit of unit id:", units.id_digit.value_counts().sort_index().to_dict())
obs_n = np.diff(np.r_[0, nwb.units.obs_intervals_index.data[:]])
print("observation intervals per unit: min", obs_n.min(), "max", obs_n.max())
obs = nwb.units.obs_intervals.data[: obs_n[0]]
print("unit 0 observation intervals equal the trial intervals:",
      bool(np.allclose(obs, tr[["start_time", "stop_time"]].values)))
allsp = np.concatenate(sess["spikes"])
ti = np.searchsorted(tr.start_time.values, allsp, side="right") - 1
inside = (ti >= 0) & (allsp <= tr.stop_time.values[np.clip(ti, 0, None)])
print("spikes inside a trial interval: %.4f" % inside.mean())
print("spike times sorted within every unit:", all(np.all(np.diff(s) >= 0) for s in sess["spikes"]))

print("\n=== behaviour ===")
t = sess["hand_t"]
dt = np.diff(t)
print("hand samples:", len(t), "| steps of 1 ms: %.4f of steps; larger steps (between trials):" % np.isclose(dt, 0.001).mean(),
      int((~np.isclose(dt, 0.001)).sum()), "| hand_vel shares timestamps:", sess["vel_shares_pos_timestamps"])
pos, vel = sess["hand_pos"], sess["hand_vel"]
print("hand position (mm): x %.1f to %.1f, y %.1f to %.1f" % (pos[:, 0].min(), pos[:, 0].max(), pos[:, 1].min(), pos[:, 1].max()))
print("hand velocity (mm/s): x %.0f to %.0f, y %.0f to %.0f" % (vel[:, 0].min(), vel[:, 0].max(), vel[:, 1].min(), vel[:, 1].max()))
ok = np.r_[np.isclose(dt, 0.001), False]
gv = np.gradient(pos[:, 0]) * 1000
m = ok & np.r_[False, ok[:-1]]
slope = np.polyfit(gv[m][::50], vel[m, 0][::50], 1)[0]
print("hand_vel x against the derivative of hand_pos x: r = %.3f, slope %.3f" % (
      np.corrcoef(gv[m][::50], vel[m, 0][::50])[0, 1], slope))

print("\n=== test file ===")
tt = nwb_test.trials.to_dataframe()
tu = nwb_test.units
print("test trials:", len(tt), "| columns:", list(tt.columns), "| split:", tt.split.unique())
print("test trial length (s):", np.unique(np.round(tt.stop_time - tt.start_time, 3)),
      "| movement onset after trial start (s):", np.unique(np.round(tt.move_onset_time - tt.start_time, 3)))
print("test units:", len(tu), "| same ids as the held-in train units:",
      set(tu.id.data[:]) == set(units.unit_id[~units.heldout]))
print("test file has behaviour:", len(nwb_test.processing) > 0)

The test file holds 574 more trials of the same session, as 0.7 s segments around movement onset
without condition labels or behaviour, and only the 137 held-in units. It cannot enter a
condition-mean analysis. Below, its trials are checked against the train file for duplicates (the same
spikes deposited twice).

In [ ]:
# duplicates: spike counts of the held-in units in the test segments against the same window of every train trial
held_ids = units.unit_id[~units.heldout].values
tid = list(tu.id.data[:])
tflat = tu.spike_times.data[:]; tidx = tu.spike_times_index.data[:]
tsp = {tid[i]: np.asarray(tflat[a:b]) for i, (a, b) in enumerate(zip(np.r_[0, tidx[:-1]], tidx))}
def count_matrix(spk_list, starts, stops):
    return np.array([np.searchsorted(s, stops) - np.searchsorted(s, starts) for s in spk_list]).T
test_counts = count_matrix([tsp[u] for u in held_ids], tt.start_time.values, tt.stop_time.values)
train_sp = [sess["spikes"][i] for i in np.where(~units.heldout)[0]]
mv = tr.move_onset_time.values
train_counts = count_matrix(train_sp, mv - 0.25, mv + 0.45)
key_train = {r.tobytes() for r in train_counts}
dup = sum(r.tobytes() in key_train for r in test_counts)
print("test segments whose spike-count vector over 137 units equals that of a train trial:", dup, "of", len(test_counts))
print("total spikes per segment: test median %d, train median %d" % (np.median(test_counts.sum(1)), np.median(train_counts.sum(1))))

### Clocks

Hand speed is checked against the movement onset of the trial table, and the spikes against the trial
events by decoding the reach direction (the active target's angle in eight sectors) from the
population spike counts in windows before and after target onset. Before target onset the direction
is not yet known and decoding must be at chance; after it, it should be above chance.

In [ ]:
p_w, v_w, err = mm.kinematic_windows(sess, tr.move_onset_time.values, 400, 600)
print("largest timestamp error of the kinematic windows (s): %.2e" % err)
spd = np.linalg.norm(v_w, axis=2)
tk = np.arange(-400, 601)
for a, b in [(-400, -300), (-200, -100), (-50, 0), (0, 50), (100, 150), (200, 300)]:
    mk = (tk >= a) & (tk < b)
    print(f"  mean hand speed {a:>5} to {b:>4} ms from movement onset: {spd[:, mk].mean():7.1f} mm/s")
peak = spd.max(1)
cross = np.array([tk[np.argmax((s > 0.1 * pk) & (tk > -300))] for s, pk in zip(spd, peak)])
print("time speed first exceeds 10%% of its peak (ms from marked onset): median %d, 5th-95th percentile %d to %d"
      % (np.median(cross), *np.percentile(cross, [5, 95])))

ang = np.degrees(np.arctan2(tr.target_y, tr.target_x))
sector = (np.round(ang / 45).astype(int) % 8).values
spk_all = sess["spikes"]
cv = StratifiedKFold(5, shuffle=True, random_state=0)
rng_clk = np.random.default_rng(1)
rows = []
for name, ev, a, b in [("before target onset", tr.target_on_time.values, -400, -100),
                       ("after target onset", tr.target_on_time.values, 100, 400),
                       ("delay, before go", tr.go_cue_time.values, -300, 0),
                       ("after movement onset", tr.move_onset_time.values, 0, 300)]:
    Xc = count_matrix(spk_all, ev + a / 1000, ev + b / 1000).astype(float)
    acc = cross_val_score(LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto"), Xc, sector, cv=cv).mean()
    accs = cross_val_score(LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto"), Xc,
                           rng_clk.permutation(sector), cv=cv).mean()
    rows.append((name, a, b, round(acc, 3), round(accs, 3)))
print(pd.DataFrame(rows, columns=["window", "from ms", "to ms", "accuracy", "accuracy, shuffled labels"]).to_string(index=False))
print("chance (largest sector share): %.3f" % (np.bincount(sector).max() / len(sector)))

### Trial selection and firing rates (M1, M2)

Trials with delay > 400 ms (M1), then the proto-trial selection under reading T2 (the two velocity
components concatenated, r > 0.9 with the condition mean; T1 set aside by the answer of 2026-10-22, and only
its count shown). Rates: spikes smoothed with a Gaussian of SD 20 ms, sampled every 10 ms, averaged over the
kept trials of each condition, locked to the go cue (-400 to +500 ms) and to movement onset (-400 to +800
ms). Tuning by reading S1 (the range over conditions; S2 set aside by the same answers).

In [ ]:
cond = tr.condition.values
C = cond.max() + 1
base = tr.delay.values > mm.MIN_DELAY_MS
mv = tr.move_onset_time.values
go = tr.go_cue_time.values
kp, kv, _ = mm.kinematic_windows(sess, mv, 150, 900)      # 1 ms, -150 to +900 ms from onset
proto = slice(0, 150 + 800 + 1)                          # M1 window, -150 to +800 ms
# reading T2 (M1, as answered 2026-10-22); T1 is set aside and only its count is shown, for the record
keep_T1 = mm.proto_trial_keep(kv[:, proto], cond, base, "T1") & base
keep = {"T2": mm.proto_trial_keep(kv[:, proto], cond, base, "T2") & base}
for T, k in (("T2", keep["T2"]), ("T1 (set aside)", keep_T1)):
    bc = np.bincount(cond[k], minlength=C)
    print(f"{T}: kept {k.sum()} of {base.sum()} trials with delay > 400 ms ({k.sum()/base.sum():.1%}); "
          f"conditions with no trial {int((bc == 0).sum())}; trials per condition min {bc[bc > 0].min()}, "
          f"median {int(np.median(bc[bc > 0]))}")

avg = {T: mm.averaging_matrix(cond, k, C) for T, k in keep.items()}
t0 = time.time()
tg, GO = mm.condition_rates(sess["spikes"], go, 400, 500, avg)
tm, MV = mm.condition_rates(sess["spikes"], mv, 400, 800, avg)
print("rates computed in %.0f s" % (time.time() - t0))

D = {}
for T in keep:
    has = np.bincount(cond[keep[T]], minlength=C) > 0
    G, M = GO[T][:, has], MV[T][:, has]
    X = mm.window_mean(G, tg, mm.PREP_WIN)      # units x conditions, preparatory (M4)
    Pm = mm.window_mean(M, tm, mm.PERI_WIN)     # peri-movement (M4)
    A = avg[T][has]
    kin_p = np.einsum("ct,tsj->csj", A, kp)
    kin_v = np.einsum("ct,tsj->csj", A, kv)
    rt_c = A @ tr.rt.values
    D[T] = dict(has=has, G=G, M=M, X=X, P=Pm, kin_p=kin_p, kin_v=kin_v, rt=rt_c,
                trc=tr[keep[T]].groupby("condition").first().loc[np.where(has)[0]])
    print(f"{T}: {has.sum()} conditions; preparatory rates {X.min():.1f} to {X.max():.1f} spikes/s; "
          f"peri-movement rates {Pm.min():.1f} to {Pm.max():.1f} spikes/s")

SEL = {}
for T in D:
    for S in ("S1",):
        pt, qt = mm.tuning(D[T]["X"], S), mm.tuning(D[T]["P"], S)
        prep = pt > mm.PREP_TUNING_MIN
        both = prep & (qt >= mm.PERI_TUNING_RATIO * pt)
        SEL[T, S] = dict(prep=prep, both=both)
        print(f"{T} {S}: preparatory tuning > 5 spikes/s: {prep.sum()} of {len(pt)} units; "
              f"also peri-movement tuning >= 1.5x: {both.sum()} "
              f"(tuning range {pt.min():.1f} to {pt.max():.1f} spikes/s)")

## recordings-jarray

> The J-array dataset holds 146 single- and multi-unit isolations from two 96-electrode arrays,
> recorded on the maze task with 108 conditions, with a mean of 2166 trials per neuron.

Expected 146 ± 14.6 units, 108 conditions (tolerance 0), 2166 ± 216.6 trials per neuron, under reading
(a) all successful trials or (b) those kept by M1. Every unit is observed on every trial (its
observation intervals are the trial intervals), so trials per neuron is the trial count.

In [ ]:
rows = [("units (all)", len(units), 146, 14.6),
        ("units (held in only)", int((~units.heldout).sum()), 146, 14.6),
        ("conditions", tr.condition.nunique(), 108, 0),
        ("trials per neuron, (a) successful trials with condition labels (train file)", len(tr), 2166, 216.6),
        ("trials per neuron, (a) all successful trials incl. 574 unlabelled test segments", len(tr) + len(tt), 2166, 216.6),
        ("trials per neuron, (b) M1 with T2", int(keep['T2'].sum()), 2166, 216.6),
        ("trials per neuron, delay > 400 ms only", int(base.sum()), 2166, 216.6)]
df = pd.DataFrame(rows, columns=["quantity", "dataset", "paper", "tolerance"])
df["within"] = (df.dataset - df.paper).abs() <= df.tolerance
print(df.to_string(index=False))

**Result: partially reproduced.** The 108 conditions match, and the 2295 labelled successful trials are
within tolerance of 2166 (reading a), as are the 1967 trials with delay > 400 ms; under M1 (reading T2) the
1786 kept trials are not. The unit count is not: the dataset has 182 units (137 held in and 45 held out),
36 more than 146 and outside the tolerance of 14.6.

## fig4d-mean-correlation-jarray

> In the J-array dataset the correlation between a neuron's preparatory and peri-movement tuning is
> positive on average but small (mean r = 0.13, p = 0.03; mean Spearman r = 0.11).

Per neuron passing M6 (reading S1), Pearson and Spearman correlations over conditions between the M4
preparatory and peri-movement rates, trials by M1 (reading T2); tests P1 to P3. Reproduced when under one
combination both means are within tolerance (0.13 ± 0.013, 0.11 ± 0.011) and p < 0.05 with the mean positive.

In [ ]:
R = {}
rows = []
for T in D:
    for S in ("S1",):
        sel = SEL[T, S]["both"]
        X, Pm = D[T]["X"][sel], D[T]["P"][sel]
        r = mm.corr_rows(X, Pm) if sel.sum() else np.array([])
        rs = mm.spearman_rows(X, Pm) if sel.sum() else np.array([])
        R[T, S] = dict(r=r, rs=rs, ids=np.where(sel)[0])
        if len(r) < 2:
            rows.append((T, S, len(r), np.nan, np.nan, "", np.nan, False))
            continue
        for P, p in mm.mean_tests(r).items():
            ok = (abs(r.mean() - 0.13) <= 0.013 + 1e-12) and (abs(rs.mean() - 0.11) <= 0.011 + 1e-12) and p < 0.05 and r.mean() > 0
            rows.append((T, S, len(r), round(r.mean(), 4), round(rs.mean(), 4), P, p, ok))
tab4d = pd.DataFrame(rows, columns=["trials", "tuning", "neurons", "mean r", "mean Spearman", "test", "p", "reproduced"])
print(tab4d.to_string(index=False))
print("\nSD of r and range, per selection:")
for k, v in R.items():
    if len(v["r"]):
        print(f"  {k}: n={len(v['r'])}, SD {v['r'].std(ddof=1):.3f}, r from {v['r'].min():.2f} to {v['r'].max():.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 2.8))
r = R["T2", "S1"]["r"]
ax.hist(r, bins=np.linspace(-1, 1, 21), color="#4a6fa5", edgecolor="white")
ax.axvline(r.mean(), color="k", lw=1)
ax.set_title(f"T2, S1: n={len(r)}, mean r={r.mean():.2f}", fontsize=9)
ax.set_xlabel("tuning correlation (all values within -1 to 1)")
ax.set_ylabel("neurons")
plt.tight_layout(); plt.show()


**Result: not reproduced.** Under T2 with S1 (17 neurons) the mean Pearson r is 0.051 and the mean Spearman
r 0.072, and no test finds the mean above zero (p 0.42 to 0.61). The means are positive but less than half
the paper's 0.13 and 0.11, outside the tolerances of 0.013 and 0.011.

## fig3-maze-most-not-maintained

> In the maze task some neurons keep similar tuning between the preparatory and peri-movement epochs, but most do not.

Neurons as for Figure 4 (M6, reading S1; trials by T2). Reproduced when at least one neuron has r ≥ 0.72 and more than
half have r < 0.72.

In [ ]:
rows = []
for (T, S), v in R.items():
    r = v["r"]
    if not len(r):
        rows.append((T, S, 0, 0, np.nan, None)); continue
    rows.append((T, S, len(r), int((r >= 0.72).sum()), round((r < 0.72).mean(), 3),
                 bool((r >= 0.72).sum() >= 1 and (r < 0.72).mean() > 0.5)))
print(pd.DataFrame(rows, columns=["trials", "tuning", "neurons", "r >= 0.72", "share r < 0.72", "holds"]).to_string(index=False))
for T in D:
    r = R[T, "S1"]["r"]
    print(f"{T} S1: largest r values", np.round(np.sort(r)[-4:], 3))

**Result: reproduced.** Under T2 with S1, one of 17 neurons keeps its tuning (r = 0.765) and 94.1% have
r < 0.72, so at least one neuron has r of 0.72 or more and more than half have r below it.

## fig4-no-separate-populations

> The distributions of tuning correlations give little evidence ... of one population with concordant
> tuning (r near 1) and another with opposing tuning (r near -1); most neurons have weakly correlated tuning.

J-array only (Figure 4D). U1: Hartigan's dip test p ≥ 0.05; U2: more neurons with |r| < 0.5 than with |r| ≥ 0.5.
Reproduced when both hold, not reproduced when neither does, methods_underspecified when they disagree.

In [ ]:
rows = []
for (T, S), v in R.items():
    r = v["r"]
    if len(r) < 4:
        rows.append((T, S, len(r), np.nan, np.nan, None, None, np.nan)); continue
    dip, p = diptest.diptest(r)
    u2 = (np.abs(r) < 0.5).sum() > (np.abs(r) >= 0.5).sum()
    rows.append((T, S, len(r), round(dip, 4), round(p, 3), p >= 0.05, bool(u2), round((np.abs(r) >= 0.8).mean(), 3)))
print(pd.DataFrame(rows, columns=["trials", "tuning", "neurons", "dip", "dip p", "U1", "U2", "share |r| >= 0.8"]).to_string(index=False))
for T in D:
    r = R[T, "S1"]["r"]
    print(f"{T} S1: |r| < 0.5 in {(np.abs(r) < 0.5).sum()}, |r| >= 0.5 in {(np.abs(r) >= 0.5).sum()}; median r {np.median(r):.3f}")

**Result: reproduced.** Under T2 with S1 the dip test finds no departure from unimodality (dip 0.071, p 0.683),
and 16 of 17 neurons have |r| < 0.5, so U1 and U2 both hold. No neuron has |r| ≥ 0.8; the median r is 0.013.

## fig4e-correlation-time-course

> The correlation between preparatory tuning and peri-movement tuning measured at single time points begins
> to decline about 200 ms before movement onset and is near zero by movement onset.

Neurons as in fig4d. Readings A1 (movement-locked rates throughout) and A2 (go-locked up to 100 ms after
the go cue, movement-locked from 300 ms before onset; in the gap between them, or where they overlap, the
two are joined linearly; each condition's go cue is placed at its mean reaction time before onset).
(i) mean correlation at onset not significantly above zero by P1 to P3; (ii) lower at onset than at -200 ms,
with more than half the drop from -400 ms to onset after -200 ms.

In [ ]:
tau = np.arange(-400, 201, 10)

def spliced(T):
    G, M, rt_c = D[T]["G"], D[T]["M"], D[T]["rt"]
    out = np.zeros(G.shape[:2] + (len(tau),))
    mv_part = M[:, :, np.searchsorted(tm, tau)]
    for c in range(G.shape[1]):
        go_end = 100 - rt_c[c]                   # end of the go-locked part, ms from onset
        g = np.array([np.interp(tau + rt_c[c], tg, G[u, c]) for u in range(G.shape[0])])
        m = mv_part[:, c]
        if go_end >= -300:     # overlap from -300 ms to go_end: fade linearly from go-locked to movement-locked
            w = np.clip((tau + 300) / max(go_end + 300, 1e-9), 0, 1)
            out[:, c] = (1 - w) * g + w * m
        else:                  # gap from go_end to -300 ms: join the two ends linearly
            g_end = G[:, c, np.searchsorted(tg, 100)][:, None]
            m_start = M[:, c, np.searchsorted(tm, -300)][:, None]
            w = np.clip((tau - go_end) / (-300 - go_end), 0, 1)
            out[:, c] = np.where(tau <= go_end, g, np.where(tau >= -300, m, (1 - w) * g_end + w * m_start))
    return out

curves = {}
rows = []
for T in D:
    print(f"{T}: mean reaction time per condition {D[T]['rt'].min():.0f} to {D[T]['rt'].max():.0f} ms")
    rate_A = {"A1": D[T]["M"][:, :, np.searchsorted(tm, tau)], "A2": spliced(T)}
    for S in ("S1",):
        ids = R[T, S]["ids"]
        if len(ids) < 2:
            continue
        for A, rates in rate_A.items():
            cc = np.array([mm.corr_rows(D[T]["X"][ids], rates[ids, :, i]) for i in range(len(tau))]).T
            mc = cc.mean(0)
            curves[T, S, A] = mc
            c0, c200, c400 = mc[tau == 0][0], mc[tau == -200][0], mc[tau == -400][0]
            tests = mm.mean_tests(cc[:, tau == 0][:, 0])
            cond_i = all((p >= 0.05) or (c0 <= 0) for p in tests.values())
            cond_ii = (c0 < c200) and ((c200 - c0) > 0.5 * (c400 - c0))
            rows.append((T, S, A, len(ids), round(c400, 3), round(c200, 3), round(c0, 3),
                         round(min(tests.values()), 3), cond_i, round((c200 - c0) / (c400 - c0), 3), cond_ii))
tab4e = pd.DataFrame(rows, columns=["trials", "tuning", "rates", "neurons", "r(-400)", "r(-200)", "r(0)",
                                    "smallest p at 0", "(i)", "share of drop after -200", "(ii)"])
print(tab4e.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
for (T, S, A), mc in curves.items():
    ax.plot(tau, mc, lw=1.5, ls="-" if A == "A1" else "--", label=f"{T} {S} {A}")
ax.axhline(0, color="0.7", lw=0.8); ax.axvline(0, color="0.7", lw=0.8)
ax.set_xlabel("ms from movement onset"); ax.set_ylabel("mean correlation with preparatory tuning")
ax.legend(fontsize=7); plt.tight_layout(); plt.show()
print("range of plotted values: %.3f to %.3f" % (min(c.min() for c in curves.values()), max(c.max() for c in curves.values())))

**Result: reproduced.** Under T2 with S1 (17 neurons) the mean correlation falls from 0.737 (A1) and 0.759 (A2)
at -400 ms and 0.608 at -200 ms to 0.012 at movement onset, where no test finds it above zero (smallest p
0.854), so (i) holds. 82.1% (A1) and 79.8% (A2) of the drop from -400 ms to onset comes after -200 ms, so (ii)
holds.

## suppfig1-population-averages

> In population averages sorted by each neuron's preparatory preference, the weak agreement ... dominates
> early, but the separation shrinks with time and shows little relation to preparatory tuning by movement onset.

Neurons as in fig4d. Per neuron the strongest and weakest preparatory conditions and those at one and two
thirds of the rank order; rates divided by the neuron's maximum rate; averaged over neurons. Separation =
strongest minus weakest. Reproduced when the preparatory separation is positive and the separation at onset
is below half (R1) or a quarter (R2) of it. The paper does not say over what the maximum is taken (answered
2026-10-22); the criterion names both readings, (a) the maximum over all conditions and times of the go- and
movement-locked averages and (b) the maximum over the four chosen conditions' traces, and both are run.

In [ ]:
rows = []
sep_curves = {}
for T in D:
    G, M, X = D[T]["G"], D[T]["M"], D[T]["X"]
    c = X.shape[1]
    for S in ("S1",):
        ids = R[T, S]["ids"]
        if len(ids) < 1:
            continue
        for NM in ("a", "b"):
            sg, sm, mids = [], [], []
            for u in ids:
                order = np.argsort(X[u])
                pick = [order[-1], order[int(round(2 * (c - 1) / 3))], order[int(round((c - 1) / 3))], order[0]]
                mx = max(G[u].max(), M[u].max()) if NM == "a" else max(G[u, pick].max(), M[u, pick].max())
                sg.append((G[u, pick[0]] - G[u, pick[3]]) / mx)
                sm.append((M[u, pick[0]] - M[u, pick[3]]) / mx)
                mids.append((M[u, pick[1]] - M[u, pick[2]]) / mx)
            sg, sm = np.mean(sg, 0), np.mean(sm, 0)
            sep_curves[T, S, NM] = (sg, sm)
            prep = sg[(tg >= -200) & (tg <= 100)].mean()
            at0 = sm[tm == 0][0]
            rows.append((T, S, NM, len(ids), round(prep, 4), round(at0, 4), round(at0 / prep, 3),
                         prep > 0 and at0 < 0.5 * prep, prep > 0 and at0 < 0.25 * prep))
print(pd.DataFrame(rows, columns=["trials", "tuning", "max", "neurons", "separation, prep epoch",
                                  "separation at onset", "ratio", "R1 (< 1/2)", "R2 (< 1/4)"]).to_string(index=False))
for T in D:
    sg, sm = sep_curves[T, "S1", "a"]
    print(f"{T} S1 (a): separation at -300, -200, -100, 0, +100 ms from onset:",
          np.round(sm[np.searchsorted(tm, [-300, -200, -100, 0, 100])], 4))

**Result: reproduced.** The separation between the strongest and weakest preparatory conditions is positive in the
preparatory epoch (0.220 of the maximum rate under normalisation a, 0.346 under b) and is 0.053 and 0.095 at
movement onset, ratios of 0.240 and 0.274. R1 (below a half) holds under both normalisations; R2 (below a
quarter) holds under (a) and not under (b). The claim needs the condition to hold under one reading.

## prep-dimensionality

> Preparatory activity is moderately high-dimensional; depending on the dataset, 7 to 10 dimensions capture 90% of its variance.

The c x n matrix of M4 preparatory rates, columns mean-centred, PCA; the number of components for 90%.
Reproduced when within 7 to 11 under one reading (N1: prep-tuned neurons, S1; N2: all neurons).

In [ ]:
rows = []
for T in D:
    X = D[T]["X"]
    for N, sel in (("N1 S1", SEL[T, "S1"]["prep"]), ("N2", np.ones(len(X), bool))):
        M_ = X[sel].T
        Mc = M_ - M_.mean(0)
        sv = np.linalg.svd(Mc, compute_uv=False) ** 2
        cum = np.cumsum(sv) / sv.sum()
        k90 = int(np.searchsorted(cum, 0.9) + 1)
        rows.append((T, N, int(sel.sum()), k90, 7 <= k90 <= 11, np.round(cum[:3], 3)))
print(pd.DataFrame(rows, columns=["trials", "neurons", "n", "components for 90%", "within 7 to 11", "first 3 cumulative"]).to_string(index=False))

**Result: not reproduced.** No reading gives 7 to 11 components. With the 83 prep-tuned neurons (N1, S1), 90% of
the variance needs 15 components; with all 182 units (N2), 18.

## PD performance (fig6 claims, fig7a, fig7b, fig7d)

Trials by T2. Neurons passing the preparatory part of M6 (S1). The peri-movement space (M5) is built for each neuron
from the condition-mean rates of all other 181 units, -150 to +800 ms from onset. The kinematic space is the
PCA of position, velocity and acceleration (acceleration as the time derivative of the dataset's hand
velocity) from -150 ms to 800 (W1) or 600 (W2) ms, each scaled to unit range, the six variables once (D1) or
repeated to 181 column blocks (D2). Task spaces by M9 from the condition-mean hand trajectories, with the
readings V1 (a: largest magnitudes; b: signed component at its largest magnitude), V2, G1 and G2.
Performance by M7 for k = 1 to 14.

In [ ]:
KMAX = 14
PERF = {}
t0 = time.time()
for T in D:
    X, M = D[T]["X"], D[T]["M"]
    win = (tm >= -150) & (tm <= 800)
    blocks = M[:, :, win]
    out = dict(blocks=blocks)
    out["peri"] = mm.peri_space_performance(blocks, X.T, KMAX)
    out["peri_start0"] = mm.peri_space_performance(M[:, :, (tm >= 0) & (tm <= 800)], X.T, KMAX)
    P_, V_ = D[T]["kin_p"], D[T]["kin_v"]
    Acc = np.gradient(V_, axis=1) * 1000
    kin, kinS = {}, {}
    for W, end in (("W1", 800), ("W2", 600)):
        sl = np.arange(0, 150 + end + 1, 10)
        for Dd, nb in (("D1", None), ("D2", blocks.shape[0] - 1)):
            Tk = mm.kinematic_tmatrix(P_[:, sl], V_[:, sl], Acc[:, sl], nb)
            kinS[W + Dd] = mm.pca_scores(Tk, KMAX)
            kin[W + Dd] = mm.space_performance_all_k(kinS[W + Dd], X.T, KMAX)
    out["kin"], out["kinS"] = kin, kinS
    feats = [mm.task_features(P_[c], V_[c], 150) for c in range(P_.shape[0])]
    trc = D[T]["trc"]
    tsp = mm.task_spaces(feats, (trc.num_barriers > 0).astype(float).values, (trc.num_targets > 1).astype(float).values)
    out["taskS"] = tsp
    out["task"] = {k: mm.loo_performance(S_, X.T) for k, S_ in tsp.items()}
    ends = np.array([f["end_ms"] for f in feats])
    print(f"{T}: end of reach {ends.min()} to {ends.max()} ms after onset (median {np.median(ends):.0f}); "
          f"reaches never falling below 5% of peak speed by +900 ms: {(ends >= 900).sum()}")
    PERF[T] = out
print("PD performance computed in %.0f s" % (time.time() - t0))

TASK_BASE = {"endpoints + max speed": ["endpoints + max speed"], "initial velocity": ["initial velocity"],
             "endpoints + orthogonal excursion": ["endpoints + orthogonal excursion"],
             "endpoints + viapoint": ["endpoints + viapoint"],
             "endpoints + max velocity": ["endpoints + max velocity (V1a)", "endpoints + max velocity (V1b)", "endpoints + max velocity (V2)"],
             "endpoints + halfpoint": ["endpoints + halfpoint"],
             "segment directions": ["segment directions (G1)", "segment directions (G2)"],
             "kitchen sink": ["kitchen sink (G1)", "kitchen sink (G2)"]}

def mean_curves(T, S):
    sel = SEL[T, S]["prep"]
    pf = PERF[T]
    return dict(n=int(sel.sum()), peri=pf["peri"][sel].mean(0), peri0=pf["peri_start0"][sel].mean(0),
                kin={k: v[sel].mean(0) for k, v in pf["kin"].items()},
                task={k: v[sel].mean() for k, v in pf["task"].items()},
                dims={k: S_.shape[1] for k, S_ in pf["taskS"].items()})

MC = {(T, S): mean_curves(T, S) for T in D for S in ("S1",)}
for (T, S), mc in MC.items():
    print(f"\n{T} {S}: {mc['n']} neurons")
    print("  k:             ", " ".join(f"{k:6d}" for k in range(1, KMAX + 1)))
    print("  peri-movement: ", " ".join(f"{v:6.3f}" for v in mc["peri"]))
    for k, v in mc["kin"].items():
        print(f"  kinematic {k}:", " ".join(f"{x:6.3f}" for x in v))
    for k, v in mc["task"].items():
        print(f"  task {k} ({mc['dims'][k]}-D): {v:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.2))
ks = np.arange(1, KMAX + 1)
mc = MC["T2", "S1"]
ax.plot(ks, mc["peri"], color="#1f4e79", lw=2, label="peri-movement")
ax.plot(ks, mc["kin"]["W1D1"], color="#c55a11", lw=2, label="kinematic (W1 D1)")
for k, v in mc["task"].items():
    ax.plot(mc["dims"][k], v, "o", color="#548235", ms=5)
ax.plot([], [], "o", color="#548235", label="task spaces")
ax.axhline(0, color="0.8", lw=0.8)
ax.set_title(f"T2, S1 ({mc['n']} neurons)", fontsize=9); ax.set_xlabel("dimensionality k")
ax.set_ylabel("mean PD performance"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()
allv = np.r_[mc["peri"], mc["kin"]["W1D1"], list(mc["task"].values())]
print("range of plotted values: %.3f to %.3f (all inside the axes)" % (allv.min(), allv.max()))


## fig6-peri-movement-space-best

> For every dataset the highest cross-validated PD performance ... is reached when the PD is expressed in
> the space built by PCA from the population peri-movement response.

Reproduced when, under every combination (K, W, D, V, G), the best mean performance over k of the
peri-movement space exceeds the best of every other space; methods_underspecified if the combinations disagree.

In [ ]:
rows = []
for (T, S), mc in MC.items():
    for K in (14, 12):
        pb, pk = mc["peri"][:K].max(), int(np.argmax(mc["peri"][:K]) + 1)
        kb = max(v[:K].max() for v in mc["kin"].values())
        tb_name = max(mc["task"], key=mc["task"].get)
        tb = mc["task"][tb_name]
        # every combination of W, D, V, G is won only if the peri-movement best exceeds every variant
        n_comb = 0; n_win = 0
        for W in ("W1", "W2"):
            for Dd in ("D1", "D2"):
                for V in ("V1a", "V1b", "V2"):
                    for Gg in ("G1", "G2"):
                        others = [mc["kin"][W + Dd][:K].max()] + [mc["task"][n] for n in mc["task"]
                                  if ("(" not in n) or (f"({V})" in n) or (f"({Gg})" in n)]
                        n_comb += 1; n_win += pb > max(others)
        rows.append((T, S, f"K{1 if K == 14 else 2}", mc["n"], round(pb, 3), pk, round(kb, 3), tb_name, round(tb, 3), f"{n_win}/{n_comb}"))
tab6 = pd.DataFrame(rows, columns=["trials", "tuning", "k max", "neurons", "peri best", "at k", "kinematic best", "best task space",
                                   "its perf.", "combinations won by peri"])
print(tab6.to_string(index=False))

**Result: reproduced.** Under T2 with S1 (83 neurons) the peri-movement space has the best performance in all 24
combinations of W, D, V and G, under K1 and K2. Its best is 0.518 at k = 14 (K1) and 0.505 at k = 12 (K2),
against 0.424 and 0.417 for the best kinematic space and 0.412 for the best task space (kitchen sink, G1).

## fig6-each-dimensionality

> Considering each dimensionality separately, PD performance is almost always higher in the peri-movement space ...;
> for the maze task the 3-D endpoints and peak speed space does not perform well.

Each task space against the peri-movement space at its dimensionality; the kinematic space against it at each
k up to K. L1: all comparisons but at most one won; L2: at least 90% won. Reproduced when the 3-D endpoints +
max speed space is below the 3-D peri-movement space and the peri-movement space wins under L1 and L2.

In [ ]:
rows = []
for (T, S), mc in MC.items():
    for K in (14, 12):
        for W in ("W1", "W2"):
            for Dd in ("D1", "D2"):
                for V in ("V1a", "V1b", "V2"):
                    for Gg in ("G1", "G2"):
                        comps = []
                        for base_name, variants in TASK_BASE.items():
                            n = variants[0] if len(variants) == 1 else next(v for v in variants if f"({V})" in v or f"({Gg})" in v)
                            d = mc["dims"][n]
                            comps.append(mc["peri"][d - 1] > mc["task"][n])
                        comps += list(mc["peri"][:K] > mc["kin"][W + Dd][:K])
                        comps = np.array(comps)
                        ems = mc["task"]["endpoints + max speed"] < mc["peri"][2]
                        L1 = (~comps).sum() <= 1
                        L2 = comps.mean() >= 0.9
                        rows.append((T, S, K, W + Dd, V, Gg, len(comps), int(comps.sum()), ems, L1, L2))
tab6b = pd.DataFrame(rows, columns=["trials", "tuning", "K", "kin", "V", "G", "comparisons", "won", "ems below peri 3-D", "L1", "L2"])
summary = tab6b.groupby(["trials", "tuning", "K"]).agg(won_min=("won", "min"), won_max=("won", "max"), comps=("comparisons", "first"),
                                                       ems=("ems below peri 3-D", "all"), L1_all=("L1", "all"), L1_any=("L1", "any"),
                                                       L2_all=("L2", "all"), L2_any=("L2", "any"))
print(summary.to_string())
for (T, S), mc in MC.items():
    print(f"{T} {S}: endpoints + max speed {mc['task']['endpoints + max speed']:.3f}, peri-movement 3-D {mc['peri'][2]:.3f}")

**Result: reproduced.** The 3-D endpoints + max speed space (0.168) is below the 3-D peri-movement space (0.271).
Under T2 with S1 the peri-movement space wins 21 of 22 comparisons (K1) and 19 of 20 (K2) in every
combination of W, D, V and G, so L1 (all but at most one) and L2 (at least 90%) both hold.

## fig6-performance-vs-dimensionality

> PD performance in PCA-based spaces generally improves with dimensionality, peaks in a middle range, and
> declines beyond about 10 dimensions through over-fitting.

Peri-movement space, k = 1 to 14: reproduced when performance at the best k exceeds that at k = 1, the best
k is 10 or below, and performance at k = 14 is below that at the best k.

In [ ]:
rows = []
for (T, S), mc in MC.items():
    p = mc["peri"]
    kb = int(np.argmax(p) + 1)
    rows.append((T, S, mc["n"], round(p[0], 3), kb, round(p.max(), 3), round(p[-1], 3),
                 p.max() > p[0], kb <= 10, p[-1] < p.max()))
print(pd.DataFrame(rows, columns=["trials", "tuning", "neurons", "k=1", "best k", "best", "k=14",
                                  "improves", "best k <= 10", "declines at 14"]).to_string(index=False))

**Result: not reproduced.** Under T2 with S1, performance rises from 0.140 at k = 1 to 0.518 at k = 14, so it
improves with dimensionality, but its best k is 14, above 10, and it does not decline at k = 14. Two of the
criterion's three conditions fail.

## fig7a-shuffled-space-poor

> A control space built from peri-movement responses in which each neuron's peri-movement pattern was inverted
> on a random half of the conditions performs poorly.

Per neuron and condition with probability 0.5, activity after -150 ms reflected about the rate at -150 ms;
10 shuffles (claims file standard choice), mean performance. H1: below the unshuffled space at every k;
H2: H1 and its best over k below the best kinematic space. Reproduced when both hold; methods_underspecified
when only H1 holds.

In [ ]:
rng7a = np.random.default_rng(7001)
SHUF = {}
for T in D:
    blocks = PERF[T]["blocks"]
    X = D[T]["X"]
    reps = []
    for s in range(10):
        flip = rng7a.random(blocks.shape[:2]) < 0.5
        b = blocks.copy()
        ref = blocks[:, :, :1]
        b[flip] = (2 * ref - blocks)[flip]
        reps.append(mm.peri_space_performance(b, X.T, KMAX))
    SHUF[T] = np.array(reps)            # shuffles x neurons x k
rows = []
for (T, S), mc in MC.items():
    sel = SEL[T, S]["prep"]
    sh = SHUF[T][:, sel].mean(1)        # shuffles x k
    shm = sh.mean(0)
    print(f"{T} {S}: shuffled mean by k", np.round(shm, 3), "| spread over shuffles of the best: %.3f to %.3f" % (sh.max(1).min(), sh.max(1).max()))
    for K in (14, 12):
        H1 = bool(np.all(shm[:K] < mc["peri"][:K]))
        kb = min(v[:K].max() for v in mc["kin"].values())
        kbmax = max(v[:K].max() for v in mc["kin"].values())
        H2_all = H1 and shm[:K].max() < kb
        rows.append((T, S, K, round(shm[:K].max(), 3), round(mc["peri"][:K].max(), 3), round(kb, 3), round(kbmax, 3), H1, H2_all))
print(pd.DataFrame(rows, columns=["trials", "tuning", "K", "shuffled best", "peri best", "kinematic best (lowest variant)",
                                  "(highest variant)", "H1", "H2 (every W, D)"]).to_string(index=False))

**Result: reproduced.** Under T2 with S1 the shuffled space (mean of 10 shuffles) is below the unshuffled
one at every k, so H1 holds. Its best, 0.227 (0.208 to 0.244 over shuffles), is below the best kinematic
space in every variant (0.421 to 0.424 under K1, 0.414 to 0.417 under K2), so H2 holds.

## fig7b-window-start-slide

> Moving the start of the peri-movement window from 150 ms before movement onset to movement onset costs little
> PD performance; at movement onset performance is 97% (monkey J) and 94% (monkeys A, B) of its original value.

Performance at the best k for the window starting at -150 and at 0 ms (end at +800). Reading J2: the J-array
ratio is compared with 0.97 ± 0.097. The A and B value cannot be tested.

In [ ]:
rows = []
for (T, S), mc in MC.items():
    for K in (14, 12):
        a, b = mc["peri"][:K].max(), mc["peri0"][:K].max()
        rows.append((T, S, K, mc["n"], round(a, 3), int(np.argmax(mc["peri"][:K]) + 1), round(b, 3),
                     int(np.argmax(mc["peri0"][:K]) + 1), round(b / a, 3), abs(b / a - 0.97) <= 0.097))
print(pd.DataFrame(rows, columns=["trials", "tuning", "K", "neurons", "best from -150", "k", "best from 0", "k", "ratio",
                                  "within 0.97 +- 0.097"]).to_string(index=False))

**Result: reproduced.** Under reading J2, the J-array ratio of performance with the window starting at onset to that
starting at -150 ms is 0.959 (K1) and 0.957 (K2) under T2 with S1, within 0.97 ± 0.097. The 94% of monkeys A
and B cannot be tested.

## fig7d-pd-consistency

> When the PD is measured on two random sets of 25 conditions, it is most stable in the peri-movement space,
> for every comparison space and all four datasets.

Per neuron, 25 conditions drawn with replacement, the PD (unit length) fitted by M7 regression, refitted on
another 25; consistency is the dot product, averaged over 100 draws (claims file standard choice) and over
neurons. Comparisons: endpoints + max speed (3-D), segment directions (6-D; G1, G2), the kinematic space at its
best k (each of W, D, K); the peri-movement space at the same dimensionality.

In [ ]:
rng7d = np.random.default_rng(7004)
NDRAW = 100

def pd_dir(S_, x, idx):
    Xd = np.column_stack([np.ones(len(idx)), S_[idx]])
    b = np.linalg.lstsq(Xd, x[idx], rcond=None)[0][1:]
    return b / np.linalg.norm(b)

def consistency(S_list, x, draws):
    # S_list: one space per neuron-independent draw set; returns mean dot product over draws
    return np.mean([pd_dir(S_list, x, i1) @ pd_dir(S_list, x, i2) for i1, i2 in draws])

rows = []
for T in D:
    X = D[T]["X"]
    c = X.shape[1]
    blocks = PERF[T]["blocks"]
    peri_S = {n: S_ for n, S_ in mm.gram_leave_one_out_scores(blocks, KMAX)}
    for S in ("S1",):
        sel = np.where(SEL[T, S]["prep"])[0]
        mc = MC[T, S]
        comps = [("endpoints + max speed", PERF[T]["taskS"]["endpoints + max speed"])]
        comps += [(f"segment directions ({g})", PERF[T]["taskS"][f"segment directions ({g})"]) for g in ("G1", "G2")]
        for K in (14, 12):
            for k_, v in mc["kin"].items():
                kb = int(np.argmax(v[:K]) + 1)
                comps.append((f"kinematic {k_} K{1 if K == 14 else 2} (k={kb})", PERF[T]["kinS"][k_][:, :kb]))
        seen = {}
        for name, Sc in comps:
            key = (name.split(" K")[0], Sc.shape[1])
            if key in seen:
                rows.append((T, S, name, Sc.shape[1], *seen[key])); continue
            k = Sc.shape[1]
            cs, cp = [], []
            for n in sel:
                draws = [(rng7d.integers(0, c, 25), rng7d.integers(0, c, 25)) for _ in range(NDRAW)]
                cs.append(consistency(Sc, X[n], draws))
                cp.append(consistency(peri_S[n][:, :k], X[n], draws))
            seen[key] = (len(sel), round(np.mean(cs), 3), round(np.mean(cp), 3), np.mean(cp) > np.mean(cs))
            rows.append((T, S, name, k, *seen[key]))
tab7d = pd.DataFrame(rows, columns=["trials", "tuning", "comparison space", "k", "neurons", "comparison consistency",
                                    "peri consistency", "peri higher"])
print(tab7d.to_string(index=False))
print("\ncomparisons won by the peri-movement space, per trials x tuning:")
print(tab7d.groupby(["trials", "tuning"])["peri higher"].agg(["sum", "count"]).to_string())

**Result: reproduced.** Under T2 with S1 the PD is more stable in the peri-movement space in all 11 comparisons:
0.752 against 0.548 for endpoints + max speed, 0.649 and 0.648 against 0.495 and 0.466 for segment directions
(G1, G2), and 0.197 to 0.319 against 0.166 to 0.230 for the kinematic spaces at their best k.

## ap-location-no-effect-on-correlation (blocked) and fig7c-anterior-posterior-performance (blocked)

Both need each unit's anterior-posterior location. The electrodes table has no coordinates (x, y, z are NaN),
and its array labels cannot be used: every unit's electrode reference points into the 96 PMd rows, so by the
file every unit is in PMd, while the dataset describes two arrays. The unit ids carry a leading digit (1 or 2)
that splits the units into 92 and 90, which looks like an array number, but nothing in the dataset says so or
says which array is which. As a variant that decides nothing, the regression of tuning correlation on that
digit is shown; its p-value does not depend on which digit is anterior.

In [ ]:
rows = []
for (T, S), v in R.items():
    if len(v["r"]) < 3:
        continue
    dig = units.id_digit.values[v["ids"]]
    res = stats.pearsonr(dig, v["r"])
    rows.append((T, S, len(v["r"]), int((dig == 1).sum()), int((dig == 2).sum()), round(res.statistic, 3), round(res.pvalue, 3)))
print(pd.DataFrame(rows, columns=["trials", "tuning", "neurons", "digit 1", "digit 2", "r (sign undetermined)", "p"]).to_string(index=False))

## fig8-model-weak-correlation

> In a simulated linear dynamical system (a harmonic oscillator held at 25 random initial states, then released),
> the preparatory to peri-movement tuning correlation of a unit is low (0.08) ...

W as printed in the Figure 8 legend (the exact rotation as a variant); initial states uniform in [-1, 1]² (I1)
or standard normal (I2); the gray interval as the first 360 (J1) or 180 (J2) steps after release. 1000 draws of
25 states. Reproduced when 0.08 lies within the central 95% of the spread under one combination.

In [ ]:
rng8a = np.random.default_rng(8001)
rows = []
for I in ("I1", "I2"):
    for J, nst in (("J1", 360), ("J2", 180)):
        for Wn, exact in (("printed W", False), ("exact rotation", True)):
            rs_ = []
            for _ in range(1000):
                x0 = rng8a.uniform(-1, 1, (25, 2)) if I == "I1" else rng8a.standard_normal((25, 2))
                traj = mm.oscillator(x0, nst, exact)
                rs_.append(np.corrcoef(x0[:, 0], traj[:, 0].mean(1))[0, 1])
            lo, hi = np.percentile(rs_, [2.5, 97.5])
            rows.append((I, J, Wn, round(np.median(rs_), 3), round(lo, 3), round(hi, 3), lo <= 0.08 <= hi))
print(pd.DataFrame(rows, columns=["initial states", "interval", "W", "median r", "2.5%", "97.5%", "0.08 inside"]).to_string(index=False))

**Result: reproduced.** Over 1000 draws of 25 initial states, the correlation spreads around zero (medians -0.041 to
0.019) with a central 95% of about -0.43 to 0.43 under every reading, and 0.08 lies inside it in all eight.
The second part of the statement (one preparatory rate leading to opposing patterns) is not tested by the
criterion.

## fig8-model-pd-perfect

> In the same simulated system, preparatory tuning is captured perfectly by a PD in the two-dimensional space
> found by PCA of the peri-movement activity.

PCA of the 25 x 2t matrix of both units' activity over the interval, k = 2; unit 1's initial rates fitted by M7
without and with cross-validation. Reproduced when R² and leave-one-out performance exceed 0.999 under every
reading; 100 draws of 25 states per reading, the smallest values shown.

In [ ]:
rng8b = np.random.default_rng(8002)
rows = []
for I in ("I1", "I2"):
    for J, nst in (("J1", 360), ("J2", 180)):
        for Wn, exact in (("printed W", False), ("exact rotation", True)):
            r2s, loos = [], []
            for _ in range(100):
                x0 = rng8b.uniform(-1, 1, (25, 2)) if I == "I1" else rng8b.standard_normal((25, 2))
                traj = mm.oscillator(x0, nst, exact)
                Tm = traj.reshape(25, -1)                 # both units over the interval
                S_ = mm.pca_scores(Tm, 2)
                y = x0[:, [0]]
                Xd = np.column_stack([np.ones(25), S_])
                fit = Xd @ np.linalg.lstsq(Xd, y, rcond=None)[0]
                r2s.append(1 - ((y - fit) ** 2).sum() / ((y - y.mean()) ** 2).sum())
                loos.append(mm.loo_performance(S_, y)[0])
            rows.append((I, J, Wn, min(r2s), min(loos), min(r2s) > 0.999 and min(loos) > 0.999))
print(pd.DataFrame(rows, columns=["initial states", "interval", "W", "smallest R2", "smallest LOO performance", "both > 0.999"]).to_string(index=False))

**Result: reproduced.** Under all eight readings the smallest R² and leave-one-out performance over 100 draws are 1.0
to the printed precision, above 0.999.

In [ ]:
print("notebook runtime: %.1f min" % ((time.time() - T_START) / 60))

## Summary

| Claim | Outcome |
|---|---|
| recordings-jarray | partially reproduced: 108 conditions and 2295 trials match; 182 units against 146 |
| recordings-single-electrode | blocked: data_missing |
| fig2b-example-concordant-tuning | blocked: data_missing |
| fig2ce-example-opposing-tuning | blocked: data_missing |
| fig3-maze-most-not-maintained | reproduced: 1 of 17 neurons with r ≥ 0.72 |
| fig4d-mean-correlation-jarray | not reproduced: mean r 0.051, Spearman 0.072, p > 0.4 |
| fig4abc-mean-correlation-single-electrode | blocked: data_missing |
| fig4-no-separate-populations | reproduced |
| fig4e-correlation-time-course | reproduced |
| suppfig1-population-averages | reproduced |
| fig6-peri-movement-space-best | reproduced: 0.518 against 0.424 (kinematic) and 0.412 (task) |
| fig6-each-dimensionality | reproduced: 21 of 22 and 19 of 20 comparisons won |
| fig6-performance-vs-dimensionality | not reproduced: best at k = 14, no decline |
| prep-dimensionality | not reproduced: 15 and 18 components, not 7 to 11 |
| fig6e-summary-across-datasets | blocked: data_missing |
| fig7a-shuffled-space-poor | reproduced: shuffled best 0.227 against 0.518 |
| fig7b-window-start-slide | reproduced (J-array ratio only) |
| ap-location-no-effect-on-correlation | blocked: metadata_insufficient |
| fig7c-anterior-posterior-performance | blocked: metadata_insufficient |
| fig7d-pd-consistency | reproduced |
| fig8-model-weak-correlation | reproduced |
| fig8-model-pd-perfect | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the
differences from the paper's method.

How the analysis differs from the paper's. Only the J-array dataset is in the archive; the speed task,
the single-electrode maze recordings and EMG are not. The session here has 182 units where the paper's J-array
dataset has 146, and nothing tells which of them the paper used, or whether the session is the same one. The
paper does not say how the two velocity components enter its proto-trial selection; the answer of
2026-10-22 fixed reading T2 (91% of trials kept, every condition; the paper says typically about 90%), and
reading T1 (74%, two conditions empty) was set aside. The paper does not define its tuning measure; the same
answers fixed reading S1, the range over conditions. Acceleration is the time derivative of the dataset's hand velocity. Movement onset is the
dataset's. All methods follow the claims file; the details it leaves open are run under every reading it names.

What was found about the dataset. Every unit's electrode reference points into the PMd rows of the
electrodes table, so by the file all 182 units are in PMd, while the dataset describes two arrays; the unit
ids' leading digit (92 and 90 units) may be the array, undocumented. The electrodes have no coordinates, so no
anterior-posterior location. The dataset holds 182 units against the paper's 146, with no record of the
paper's unit or trial selection. 574 trials sit in a test file as 0.7 s segments with no condition labels or
behaviour and only the 137 held-in units. 14 trials have a negative delay (down to -3 ms). The behaviour
series' timestamps run on a 1 ms grid within trials with gaps between trials, and the spike times only cover
the trials; both share the trial table's clock as far as the checks show (hand speed rises at the marked
movement onset; reach direction is decoded from spikes after target onset, 0.661, and not before, 0.166).